Hotel Booking EDA
El análisis exploratorio de datos (EDA) para el conjunto de datos de reservas hoteleras nos permitirá conocer a fondo la información contenida en el dataset. En esta primera etapa, exploraremos aspectos clave como:

Patrones de cancelación.
Datos faltantes y posibles valores atípicos.
Distribución de las variables.
Relación entre variables.
Selección de variables relevantes para el análisis posterior.
Esta aproximación inicial nos ayudará a entender la estructura y calidad de los datos, sentando las bases para un análisis más profundo y la identificación de insights relevantes para el negocio.

Aquí veremos una APROXIMACIÓN al objetivo del trabajo, pero también incluye operaciones que hemos comentado en clase. Los EDAs no son todos iguales.



Set Up
here we set up the environment and import the required packages.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import zscore, f_oneway
import warnings
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import TargetEncoder, OneHotEncoder
from sklearn.impute import KNNImputer
from scipy.stats import chi2_contingency


warnings.filterwarnings("ignore")

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
df_raw = pd.read_csv('../data/hotel_bookings.csv')

In [ ]:
df_raw.info()

In [ ]:
numeric_cols = [col for col in df_raw.columns if df_raw[col].dtype in ['int64', 'float64']]
str_cols = [col for col in df_raw.columns if df_raw[col].dtype == 'object']

print(f'variables numericas {numeric_cols}')
print(f'-----------------------------------------------')
print(f'variables categoricas {str_cols}')

In [ ]:
df_raw.describe(include=[np.number])

In [ ]:
# modificamos la variable is_canceled a categorica
df_raw['is_canceled'] = df_raw['is_canceled'].astype('object')

# modificamos la variable agent a categorica
df_raw['agent'] = df_raw['agent'].astype('object')

# drop is_canceled from numeric_cols
numeric_cols.remove('is_canceled')
# drop agent from numeric_cols
numeric_cols.remove('agent')
str_cols.append('is_canceled')
str_cols.append('agent')
df_raw.describe(include='object')

In [ ]:
# vemos cuantas veces se ha cancelado la reserva

n_cancelled = df_raw['is_canceled'].value_counts()

print(f'para el total de {len(df_raw)} reservas, {n_cancelled[1]} han sido canceladas y {n_cancelled[0]} no han sido canceladas,' \
    f' lo que representa un {n_cancelled[1] / len(df_raw) * 100:.2f}% del total de reservas')

Análisis Univariado
Columnas numéricas

In [ ]:
fig, axes = plt.subplots(nrows=len(df_raw.select_dtypes(include=[np.number]).columns), ncols=2, figsize=(15, 5 * len(df_raw.select_dtypes(include=[np.number]).columns)))

for i, column in enumerate(df_raw[numeric_cols]):
    # Histograma de densidad
    sns.histplot(df_raw[column], kde=True, ax=axes[i, 0])
    axes[i, 0].set_title(f'Histograma de densidad de {column}')
    
    # Boxplot
    sns.boxplot(x=df_raw[column], ax=axes[i, 1])
    axes[i, 1].set_title(f'Boxplot de {column}')

plt.tight_layout()
plt.show()

In [ ]:
# vuelvo a hacer un describe de las variables numéricas con baja cardinalidad con mayor detalle de los percentiles

numeric_cols_low_cardinality = [col for col in numeric_cols if df_raw[col].nunique() < 50]

df_raw[numeric_cols_low_cardinality].describe(percentiles=[.25, .5, .75, .9, .95, .99]).T.sort_values(by='50%', ascending=False).style.background_gradient(cmap='coolwarm')

In [ ]:
df_low_c_logged = df_raw[numeric_cols_low_cardinality].copy()
# aplico logaritmo a las variables con baja cardinalidad
for col in df_low_c_logged.columns:
    if df_low_c_logged[col].min() > 0:
        df_low_c_logged[col] = np.log(df_low_c_logged[col])
    else:
        df_low_c_logged[col] = np.log(df_low_c_logged[col] + 1)

In [ ]:
# Visualizo solo el histograma de las variables con baja cardinalidad

fig, axs = plt.subplots(nrows=len(df_low_c_logged.columns), ncols=1, figsize=(10, 5 * len(df_low_c_logged.columns)))

for i, column in enumerate(df_low_c_logged.columns):
    # Histograma de densidad
    sns.histplot(df_low_c_logged[column], kde=True, ax=axs[i])
    axs[i].set_title(f'Histograma de densidad de {column}')

Variables Categóricas

ahora vamos a hacer un análisis univariado de las variables categóricas. Para ello, vamos a utilizar la función value_counts() de pandas para contar el número de ocurrencias de cada categoría en cada variable categórica. Luego, vamos a graficar los resultados utilizando la función plot() de pandas.

In [ ]:
fig, axs = plt.subplots(nrows=len(str_cols) // 2 + len(str_cols) % 2, ncols=2, figsize=(20, 5 * (len(str_cols) // 2 + len(str_cols) % 2)))

# Aplanar el array de ejes para iterar fácilmente
axs = axs.flatten()

for i, col in enumerate(str_cols):
    sns.countplot(data=df_raw, x=col, order=df_raw[col].value_counts().index, ax=axs[i])
    axs[i].set_title(f'Distribución de {col}')
    axs[i].set_xlabel(col)
    axs[i].set_ylabel('Frecuencia')
    axs[i].tick_params(axis='x', rotation=45)  # Rotar las etiquetas del eje x para mejor legibilidad

# Eliminar subplots vacíos si el número de columnas es impar
if len(str_cols) % 2 != 0:
    fig.delaxes(axs[-1])

plt.tight_layout()
plt.show()

Análisis Bivariado contra la variable objetivo

In [ ]:
target = 'is_canceled'

fig, axs = plt.subplots(nrows=len(numeric_cols), ncols=1, figsize=(10, 5 * len(numeric_cols)))

# Graficar cada columna numérica contra el target en un subplot
for i, col in enumerate(numeric_cols):
    sns.kdeplot(data=df_raw[df_raw[target] == df_raw[target].unique()[0]], x=col, ax=axs[i], label=f'{target} = {df_raw[target].unique()[0]}', fill=True)
    sns.kdeplot(data=df_raw[df_raw[target] == df_raw[target].unique()[1]], x=col, ax=axs[i], label=f'{target} = {df_raw[target].unique()[1]}', fill=True)
    axs[i].set_title(f'Densidad de {col} por {target}')
    axs[i].set_xlabel(col)
    axs[i].set_ylabel('Densidad')
    axs[i].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Vamos a llevar a cabo la windsorizacion de las variables de baja cardinalidad a ver como se comporta en el analisis bivariable
def windsorize_upper(df, columns, upper_percentile=0.99):
    """
    Windsorize the upper tail of a series at a given percentile.
    """
    for col in columns:
        upper_limit = df[col].quantile(upper_percentile)
        df[col] = np.where(df[col] > upper_limit, upper_limit, df[col])
    return df


df_windsorized = windsorize_upper(df_raw.copy(), numeric_cols_low_cardinality)


fig, axs = plt.subplots(nrows=len(numeric_cols_low_cardinality), ncols=1, figsize=(10, 5 * len(numeric_cols_low_cardinality)))
for i, col in enumerate(numeric_cols_low_cardinality):
    sns.kdeplot(data=df_windsorized[df_windsorized[target] == df_windsorized[target].unique()[0]], x=col, ax=axs[i], label=f'{target} = {df_windsorized[target].unique()[0]}', fill=True)
    sns.kdeplot(data=df_windsorized[df_windsorized[target] == df_windsorized[target].unique()[1]], x=col, ax=axs[i], label=f'{target} = {df_windsorized[target].unique()[1]}', fill=True)
    axs[i].set_title(f'Densidad de {col} por {target} (Windsorizado)')
    axs[i].set_xlabel(col)
    axs[i].set_ylabel('Densidad')
    axs[i].legend()

plt.tight_layout()
plt.show()

In [ ]:
df_raw.describe(include=[np.number], percentiles=[.25, .5, .75, .9, .95, .99]).T.sort_values(by='50%', ascending=False).style.background_gradient(cmap='coolwarm')

In [ ]:
# Para aquellos valores que tienen el 90% de los valores en un solo valor, vemos la distribución de valores únicos y decidimos si son susceptibles de ser eliminados

for col in df_raw[numeric_cols_low_cardinality].columns:
    most_common_value = df_raw[col].value_counts().idxmax()
    filtered_data = df_raw[col][df_raw[col] != most_common_value]
    plt.figure()
    filtered_data.value_counts().plot(kind='bar')
    plt.title(f'{col} (excluding most common value)')
    plt.show()

In [ ]:
# Vamos a ver como se distribuyen las variables categoricas con respecto a la variable target
fig, axs = plt.subplots(nrows=len(str_cols), ncols=1, figsize=(10, 5 * len(str_cols)))

for i, col in enumerate(str_cols):
    sns.countplot(data=df_raw, x=col, hue=target, ax=axs[i], order=df_raw[col].value_counts().index)
    axs[i].set_title(f'Distribución de {col} por {target}')
    axs[i].set_xlabel(col)
    axs[i].set_ylabel('Frecuencia')
    axs[i].tick_params(axis='x', rotation=45)  # Rotar las etiquetas del eje x para mejor interpretación
    axs[i].legend(title=target)

plt.tight_layout()

In [ ]:
# Vamos a hacer análisis bivariado un poco más específico

canceled_perc = df_raw['is_canceled'].value_counts(normalize=True)
canceled_perc

In [ ]:
# ¿Se cancela más en un tipo de hotel que en otro?

plt.figure(figsize=(8, 4))

# Create the countplot
ax1 = sns.countplot(x='hotel', hue='is_canceled', data=df_raw)
legend_labels, _ = ax1.get_legend_handles_labels()
ax1.legend(bbox_to_anchor=(1, 1))

total = len(df_raw)
for p in ax1.patches:
    percentage = '{:.2f}%'.format(100 * p.get_height() / total)
    x = p.get_x() + p.get_width() / 2 - 0.05
    y = p.get_height()
    ax1.annotate(percentage, (x, y), ha='center', va='bottom')

plt.title('Reservation status in different hotels', size=20, color='Black')
plt.xlabel('Hotel', color='Black')
plt.ylabel('Number of Reservations', color='Black')
plt.legend(['Not Cancelled', 'Cancelled'])

In [ ]:
# ¿Las cancelaciones tienen estacionalidad?

df_raw['month']=pd.to_datetime(df_raw['reservation_status_date']).dt.month
plt.figure(figsize=(16,8))
ax1 = sns.countplot(x='month', hue='is_canceled', data= df_raw)
legend_lebels,_ = ax1.get_legend_handles_labels()
plt.title('Reservation Status Per Month', size = 20)
plt.xlabel('month')
plt.ylabel('Number of Reservation')
plt.legend(['Not Canceled','Canceld'])

In [ ]:
# ¿Tiene relación de la variable de tarifa diaria con la cancelación?
 
fig, axes = plt.subplots(1, 2, figsize=(15, 8))

sns.kdeplot(
    data=df_raw[df_raw[target] == df_raw[target].unique()[0]],
    x='adr', label=f'{target} = {df_raw[target].unique()[0]}',
    fill=True, ax=axes[0],
    color='blue'
    )
axes[0].set_title('Density of ADR for Not Canceled')
axes[0].set_xlabel('ADR')
axes[0].set_ylabel('Density')
axes[0].legend()

# Segundo gráfico
sns.kdeplot(
    data=df_raw[df_raw[target] == df_raw[target].unique()[1]],
    x='adr', label=f'{target} = {df_raw[target].unique()[1]}',
    fill=True, ax=axes[1],
    color='red'
    )
axes[1].set_title('Density of ADR for Canceled')
axes[1].set_xlabel('ADR')
axes[1].set_ylabel('Density')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# ¿Cuales son los paises que más cancelan - Top 10?

top_10_countries = df_raw['country'].value_counts().head(10).index
top_10_countries_df = df_raw[df_raw['country'].isin(top_10_countries)]
plt.figure(figsize=(15, 8))
sns.countplot(data=top_10_countries_df, x='country', hue='is_canceled')
plt.title('Top 10 Countries with Most Cancellations')
plt.xlabel('Country')
plt.ylabel('Number of Reservations')
plt.xticks(rotation=45)
plt.legend(title='Reservation Status', labels=['Not Canceled', 'Canceled'])
plt.show()
# ¿Cual es el canal de reserva que más cancela?
plt.figure(figsize=(15, 8))
sns.countplot(data=df_raw, x='distribution_channel', hue='is_canceled')
plt.title('Reservation Status by Distribution Channel')
plt.xlabel('Distribution Channel')
plt.ylabel('Number of Reservations')
plt.xticks(rotation=45)
plt.legend(title='Reservation Status', labels=['Not Canceled', 'Canceled'])
plt.show()

Nulos y Atípicos

In [ ]:
# Tenemos que quitar el .0 de agent que tiene sintaxis float erroneamente
# comprobamos si existe el 0 
df_clean = df_raw.copy()
print(f"¿Existe el agente 0? {(df_clean['agent'] == 0).any()}")

# convertimos a 0
df_clean['agent'] = df_clean['agent'].fillna(0)
# convertimos a int
df_clean['agent'] = df_clean['agent'].astype(int)
# convertimos a categorical
df_clean['agent'] = df_clean['agent'].astype(str)
# volvemos a convertir a nulo para análisis
df_clean['agent'] = df_clean['agent'].replace('0', np.nan)

In [ ]:
# Imputación de valores faltantes

# Listamos las columnas con valores nulos
df_missing = df_clean.isnull().sum()
df_missing = df_missing[df_missing > 0]
df_missing = df_missing.reset_index()

# Calculamos el porcentaje de valores nulos 
df_missing['percentage'] = df_missing[0] / len(df_clean) * 100
df_missing

Decisiones sobre nulos.
Los nulos de children son mínimos, por lo que imputaremos con la moda.
Los nulos de country son pocos, imputaremos con la moda.
Agent tiene un número significativo de nulos, por lo que crearemos una categoría "desconocido" para los nulos.
Company tiene un número muy importante de nulos, por lo que la mejor opción es eliminar la columna.

In [ ]:
# Imputamos los valores nulos de las variables categóricas con la moda
df_clean['children'].fillna(df_clean['children'].mode()[0], inplace=True)
df_clean['country'].fillna(df_clean['country'].mode()[0], inplace=True)
df_clean['agent'].fillna('unknown', inplace=True)
df_clean.drop(columns=['company'], inplace=True)
numeric_cols.remove('company')

Tratamiento de Outliers

In [ ]:
df_clean[numeric_cols].head()

Alta cardinalidad

In [ ]:
# Vemos cardinalidad

df_clean[numeric_cols].nunique().sort_values(ascending=False)

# Calculamos rango intercuartilico para aquellas variables con más de 50 valores únicos
high_cardinality = [col for col in df_clean[numeric_cols].columns if df_clean[col].nunique() > 50]

df_clean_num_high_cardinality = df_clean[high_cardinality]

high_cardinality

In [ ]:
# Definir el número de columnas
ncols = 3
nrows = (len(df_clean_num_high_cardinality.columns) + ncols - 1) // ncols

fig, axs = plt.subplots(nrows=nrows, ncols=ncols, figsize=(15, 5 * nrows))

for i, column in enumerate(df_clean_num_high_cardinality.columns):
    row = i // ncols
    col = i % ncols
    sns.histplot(df_clean_num_high_cardinality[column], kde=True, ax=axs[row, col])
    axs[row, col].set_title(f'Histograma de densidad de {column}')
    axs[row, col].set_xlabel(column)
    axs[row, col].set_ylabel('Frecuencia')

plt.tight_layout()
plt.show()

In [ ]:
df_clean_num_high_cardinality.quantile(0.75) - df_clean_num_high_cardinality.quantile(0.25)

In [ ]:
iqr = df_clean_num_high_cardinality.quantile(0.75) - df_clean_num_high_cardinality.quantile(0.25)
# aplicamos a alta cardinalidad a df_raw_num_high_cardinality

def iqr_windsorize(df, columns, iqr_threshold=1.5, windsorize_threshold=0.95):
    """
    Imputación de valores atípicos utilizando el rango intercuartílico (IQR) si el rango intercuartílico es distinto de 0.
    Si el rango intercuartílico es 0, se aplicará la windsorización a los valores superiores al percentil 95.
    :param df: DataFrame de entrada.
    :param columns: Lista de columnas a procesar.
    :return: DataFrame con los valores atípicos imputados.
    """
    for col in columns:
        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        if q1 != q3:
            iqr_value = q3 - q1
            lower_bound = q1 - iqr_threshold * iqr_value
            upper_bound = q3 + iqr_threshold * iqr_value
            df[col] = np.where(df[col] < lower_bound, lower_bound, df[col])
            df[col] = np.where(df[col] > upper_bound, upper_bound, df[col])
        else:
            # Si el rango intercuartílico es 0, aplicamos la windsorización a los valores superiores al percentil 95
            upper_bound = df[col].quantile(windsorize_threshold)
            df[col] = np.where(df[col] > upper_bound, upper_bound, df[col])
    return df

df_clean_num_high_cardinality = iqr_windsorize(df_clean_num_high_cardinality, high_cardinality)
print('columnas con alta cardinalidad antes de la imputación')
df_clean[high_cardinality].describe(percentiles=[.25, .5, .75, .9, .95, .99]).T.sort_values(by='50%', ascending=False).style.background_gradient(cmap='CMRmap')

In [ ]:
print('columnas con alta cardinalidad después de la imputación')
df_clean_num_high_cardinality.describe(percentiles=[.25, .5, .75, .9, .95, .99]).T.sort_values(by='50%', ascending=False).style.background_gradient(cmap='coolwarm')

In [ ]:
#Como vemos, las primeras tres columnas parecen haberse imputado correctamente, pero previous bookings y days in waiting list no. Vamos a realizar un análisis univariado de estas dos variables para ver como se comportan los outliers.

prev_book = df_clean['previous_bookings_not_canceled']

prev_book.loc[prev_book > 0] = 1
prev_book.loc[prev_book == 0] = 0
# Análisis bivariado contra target de prev_book vs is_canceled en porcentaje
plt.figure(figsize=(10, 5))
ax = sns.countplot(x=prev_book, hue=df_raw['is_canceled'])

total = len(df_raw)
for p in ax.patches:
    percentage = '{:.1f}%'.format(100 * p.get_height() / total)
    x = p.get_x() + p.get_width() / 2 - 0.05
    y = p.get_height()
    ax.annotate(percentage, (x, y), ha='center', va='bottom')

plt.title('Previous Bookings vs Cancellations')
plt.xlabel('Previous Bookings')
plt.ylabel('Count')
plt.legend(title='Reservation Status', labels=['Not Canceled', 'Canceled'])

plt.show()

In [ ]:
#La variable en forma binaria aporta información ya cuando existen cancelaciones previas no parece que haya tantas cancelaciones, por lo que para eliminar los outliers de esta variable vamos a optar por tratarla como una variable categórica. Para ello, vamos a crear una nueva variable que tome el valor 1 si hay cancelaciones previas y 0 si no las hay.
waiting_list = df_raw['days_in_waiting_list']
waiting_list.loc[waiting_list > 0] = 1
waiting_list.loc[waiting_list == 0] = 0
print(waiting_list.value_counts(normalize=True))    
"""Viendo este porcentaje, es recomendable hacer lo mismo que en el paso anterior con previous_bookings_not_canceled,
es decir, convertir la variable en binaria y luego hacer el análisis bivariado con respecto a la variable target"""

# Análisis bivariado contra target de waiting_list vs is_canceled en porcentaje
plt.figure(figsize=(10, 5))
ax = sns.countplot(x=waiting_list, hue=df_raw['is_canceled'])
total = len(df_raw)
for p in ax.patches:
    percentage = '{:.1f}%'.format(100 * p.get_height() / total)
    x = p.get_x() + p.get_width() / 2 - 0.05
    y = p.get_height()
    ax.annotate(percentage, (x, y), ha='center', va='bottom')

plt.title('waiting_list vs Cancellations')
plt.xlabel('waiting_list')
plt.ylabel('Count')
plt.legend(title='waiting_list Status', labels=['Not Canceled', 'Canceled'])

In [ ]:
#En esta visualización podemos ver que si aporta información, ya que la mayoría de los clientes que han cancelado no tienen cancelaciones previas. Por lo tanto, vamos convertirla en una variable categórica binaria.

# Hacemos las tres transformaciones a la vez
df_raw['previous_bookings_not_canceled'] = prev_book
df_raw['days_in_waiting_list'] = waiting_list
# eliminamos de high cardinality estas dos variables
high_cardinality.remove('previous_bookings_not_canceled')
high_cardinality.remove('days_in_waiting_list')

# eliminamos de numeric_cols estas dos variables
numeric_cols.remove('previous_bookings_not_canceled')
numeric_cols.remove('days_in_waiting_list')

# Imputamos con IQR
df_raw = iqr_windsorize(df_raw, high_cardinality)
# boxplots 
fig, axs = plt.subplots(nrows=len(high_cardinality), ncols=1, figsize=(10, 5 * len(high_cardinality)))
for i, column in enumerate(high_cardinality):
    # Boxplot
    sns.boxplot(x=df_clean_num_high_cardinality[column], ax=axs[i])
    axs[i].set_title(f'Boxplot de {column}')
    axs[i].set_xlabel(column)
    axs[i].set_ylabel('Value')
plt.tight_layout()
plt.show()

Decisiones con outliers de baja cardinalidad

In [ ]:
# Volvemos a leer las columnas de baja cardinalidad
low_cardinality_under50 = [col for col in df_clean.columns if df_clean[col].dtype in ['int64', 'float64'] and df_clean[col].nunique() < 50]
# eliminamos las binarias
low_cardinality_under50.remove('previous_bookings_not_canceled')
# low_cardinality_under50.remove('days_in_waiting_list')
low_cardinality_under50.remove('is_repeated_guest')
df_clean[low_cardinality_under50].nunique().sort_values(ascending=False)

In [ ]:
# Vamos a ver la asimetría de las variables de baja cardinalidad
print('__________asimetría de las variables de baja cardinalidad__________')
print(df_clean[low_cardinality_under50].skew().sort_values(ascending=False))
# Vemos la curtosis de las variables de baja cardinalidad
print('__________curtosis de las variables de baja cardinalidad__________')
print(df_clean[low_cardinality_under50].kurtosis().sort_values(ascending=False))

In [ ]:
# seleccionamios estas variables
low_cardinality_high_skew = df_clean[low_cardinality_under50].skew().sort_values(ascending=False)[0:4].index.tolist()

df_clean[low_cardinality_high_skew].describe(percentiles=[.25, .5, .75, .9, .95, .99]).T.sort_values(by='50%', ascending=False).style.background_gradient(cmap='coolwarm')

In [ ]:
"""atendiendo a estos datos, vamos a eliminar los valores que superen el p99, ya que no tienen sentido en el contexto del negocio. No tiene sentido imputar por la mediana
porque si atendemos al concepto del negocio, se trata de clientes muy particulares a nivel adultos (grupos), de niños (familias extremadamente numerosas)"""
df_hypo_q99_plus = df_clean[low_cardinality_high_skew]

# Iterar sobre cada columna en low_cardinality_hgh_skew y eliminar los valores que superen el percentil 99
for col in low_cardinality_high_skew:
    p99 = df_hypo_q99_plus[col].quantile(0.99) + 3
    df_hypo_q99_plus = df_hypo_q99_plus[df_hypo_q99_plus[col] <= p99]

# Mostrar los primeros 10 registros del DataFrame filtrado
print(f'elementos eliminados: {df_clean.shape[0] - df_hypo_q99_plus.shape[0]}')

df_hypo_q99_plus.describe(percentiles=[.25, .5, .75, .9, .95, .99]).T.sort_values(by='50%', ascending=False).style.background_gradient(cmap='coolwarm')

Como podemos ver, después de hacer una prueba iterativa, hemos considerado que eliminar los valores por encima del p99 + 3 es lo más adecuado. Si consideramos que en algún momento estos outliers pueden afectar al modelo, podemos aplicar un tratamiento diferente, como la transformación logarítmica o la transformación Box-Cox. Sin embargo, en este caso, vamos a optar por dejarlo de esta forma

Importante!!!! Cabe destacar que el outliers inferiores no son muy significativos, por lo que no los vamos a eliminar. En algunos casos, como el valor 0 de adultos, puede despertar alguna duda, porque no es común una reserva sin adultos. Para eliminar esos datos, que reportan ~400 casos, deberíamos conocer bien el contexto del negocio y entender porqué puede ser 0 ese valor. Si se trata de un error, podríamos eliminarlo, pero si se trata de una reserva diferente (ej, gubernamental) no sería correcto eliminarlo. En este caso, vamos a dejarlo como está.

In [ ]:
# Eliminamos el indice de los registros eliminados
df_clean = df_clean[df_clean.index.isin(df_hypo_q99_plus.index)]


# Aceptamos el experimento, modificamos los valores
for col in low_cardinality_high_skew:
    df_clean[col] = df_hypo_q99_plus[col]

Análisis Multivariante

In [ ]:
 # Hacemos Pairplot para las variables numéricas 

sns.pairplot(df_clean[numeric_cols], diag_kind='kde')
plt.show()

In [ ]:
numeric_cols_with_target = numeric_cols.copy()
numeric_cols_with_target.append('is_canceled')

corr = df_clean[numeric_cols_with_target].corr()
corr_target = corr['is_canceled'].sort_values(ascending=False)
corr_target

In [ ]:
# Análizamos la relación entre variables categóricas mediante chi-cuadrado
def chi_square_test(df, col1, col2):
    contingency_table = pd.crosstab(df[col1], df[col2])
    chi2, p, _, _ = chi2_contingency(contingency_table)
    return chi2, p
def analyze_categorical_relationships(df, categorical_cols):
    results = []
    for i in range(len(categorical_cols)):
        for j in range(i + 1, len(categorical_cols)):
            col1 = categorical_cols[i]
            col2 = categorical_cols[j]
            chi2, p = chi_square_test(df, col1, col2)
            results.append((col1, col2, chi2, p))
    return results
categorical_relationships = analyze_categorical_relationships(df_clean, str_cols)
results_df = pd.DataFrame(categorical_relationships, columns=['col1', 'col2', 'chi2', 'p-value'])
results_df['significant'] = results_df['p-value'] < 0.05

results_df['p-value'] = results_df['p-value'].apply(lambda x: '{:.2e}'.format(x))
results_df['chi2'] = results_df['chi2'].apply(lambda x: '{:.2f}'.format(x))
results_df.sort_values(by='chi2', ascending=False, inplace=True)
results_df

In [ ]:
def cramers_v(x, y):
    """
    Calcula Cramér's V para dos variables categóricas.
    
    Parámetros:
    - x, y: Series o arrays de pandas que representan variables categóricas.
    
    Retorna:
    - Valor de Cramér's V, entre 0 y 1.
    """
    tabla_cruzada = pd.crosstab(x, y)

    # calculamos chi2, el p valor asociado, los grados de libertad y la tabla esperada
    chi2, p, dof, expected = chi2_contingency(tabla_cruzada)
    #hacemos la suma de la tabla cruzada global para obetener el nº total de datos y normalizar 
    n = tabla_cruzada.sum().sum()
    min_dim = min(tabla_cruzada.shape) - 1 # Cálculo de la dimensión mínima
    if min_dim == 0:
        return np.nan # si una de la variable tiene solo un valor, no se puede calcular la v de cramer. Habría que eliminarla
    
    return np.sqrt(chi2 / (n * min_dim))

# Calcular Cramér's V para cada par de variables categóricas
cramers_v_results = []
for i in range(len(str_cols)):
    for j in range(i + 1, len(str_cols)):
        col1 = str_cols[i]
        col2 = str_cols[j]
        cv = cramers_v(df_clean[col1], df_clean[col2])
        cramers_v_results.append((col1, col2, cv))
cramers_v_df = pd.DataFrame(cramers_v_results, columns=['col1', 'col2', 'Cramér\'s V'])
cramers_v_df['Cramér\'s V'] = cramers_v_df['Cramér\'s V'].apply(lambda x: '{:.2f}'.format(x))
cramers_v_df.sort_values(by='Cramér\'s V', ascending=False, inplace=True)
cramers_v_df.head(30)

In [ ]:
# Eliminamos las columnas de status
df_clean.drop(columns=['reservation_status', 'reservation_status_date'], inplace=True)
str_cols.remove('reservation_status')
str_cols.remove('reservation_status_date')
# Son demasiadas variables, vamos a hacer un pairplot solo con las variables que tienen una correlación mayor a 0.2 con la variable target
numeric_cols_with_target = numeric_cols.copy()
numeric_cols_with_target.append('is_canceled')

corr = df_clean[numeric_cols_with_target].corr()
corr_target_selected = corr['is_canceled'].sort_values(ascending=False)
corr_target_selected = corr_target_selected[abs(corr_target_selected) > 0.2]
print(corr_target_selected)
# Filtramos las variables que tienen una correlación mayor a 0.5 con la variable target
numeric_cols_target = corr_target_selected.index.tolist()
# Hacemos Pairplot para las variables numéricas que tienen una correlación mayor a 0.5 con la variable target
sns.pairplot(df_clean[numeric_cols_target], diag_kind='kde')
plt.show()

In [ ]:
# Test Anova
group1 = df_clean[df_clean['is_canceled'] == 0]['adr']
group2 = df_clean[df_clean['is_canceled'] == 1]['adr']

# Perform the ANOVA test
anova_result = f_oneway(group1, group2)

print('ANOVA test results:')
print(f'F-statistic: {anova_result.statistic}')
print(f'p-value: {anova_result.pvalue}')

"""
En este caso, el p-valor es menor que 0.05, lo que indica que hay diferencias significativas en las medias de los grupos, 
por lo que podemos rechazar la hipótesis nula de que las medias son iguales. 
Esto sugiere que la tarifa diaria (adr) tiene un efecto significativo en la cancelación de reservas."""

In [ ]:
# ___________Test de información mutua, hemos importado clasiff al ser la variable objetivo categórica.___________


# A Agent, country le aplicamos target encoding debido a su alta cardinalidad, mientras que el resto de variables categóricas las convertimos a dummies
df_selection = df_clean.copy()
# Aplicamos target encoding a las variables categóricas con alta cardinalidad
target_encoder = TargetEncoder()
df_selection['is_canceled'] = df_selection['is_canceled'].astype(int)
df_selection['agent'] = target_encoder.fit_transform(df_selection[['agent']], df_selection['is_canceled'])
df_selection['country'] = target_encoder.fit_transform(df_selection[['country']], df_selection['is_canceled'])
# Convertimos las variables categóricas de baja cardinalidad a dummies
str_cols_dummies = [col for col in str_cols if col not in ['agent', 'country']]
df_dummies = pd.get_dummies(df_selection[str_cols_dummies], drop_first=True, prefix_sep='-')
df_dummies = df_dummies.replace({True: 1, False: 0})
# Concatenamos las variables dummies con el resto de variables 
df_selection = pd.concat([df_selection.drop(columns=str_cols_dummies), df_dummies], axis=1)
# Añadimos las variables numéricas
df_selection = pd.concat([df_selection, df_clean[numeric_cols]], axis=1)
del df_dummies

In [ ]:
X = df_selection.drop(columns=['is_canceled']) # vector de características
y = df_selection['is_canceled'] # vector objetivo

mi_scores = mutual_info_classif(X, y, random_state=42)
mi_series = pd.Series(mi_scores, index=X.columns).sort_values(ascending=False)

In [ ]:
mi_series

# separamos columnas y valores y sumamos importancia
mi_series_grouped = mi_series.groupby(lambda x: x.split('-')[0]).sum().reset_index()
mi_series_grouped['type_column'] = mi_series_grouped['index'].apply(
    lambda col: 'numeric' if col in numeric_cols else 'categorical' if col in str_cols else 'other'
)
mi_series_grouped.sort_values(by=0, ascending=False).reset_index(drop=True).rename(columns={'index': 'col_name', 0: 'importance'}).style.background_gradient(cmap='coolwarm')

In [ ]:
# eliminamos las variables que no tienen importancia
cols_to_keep = mi_series_grouped[mi_series_grouped[0] > 0.005]['index'].tolist()

df_processed = df_clean[cols_to_keep]

# añadimos variables numéricas
df_processed = pd.concat([df_processed, df_clean['is_canceled']], axis=1)

In [ ]:
# Ahora eliminamos las variables numéricas que no tienen importancia 

# hemos visto que muchas variables prácticamente no tienen correlación con la variable target, por lo que llevaremos a cabo un último análisis para decidir si eliminarlas o no
# Este análisis será Spearman, ya que no se distribuyen normalmente
X = df_processed.drop(columns=['is_canceled']) # vector de características
X_numeric = X.select_dtypes(include=[np.number])    
y = df_processed['is_canceled'] # vector objetivo

# Calculamos la correlación de Spearman
spearman_corr = X_numeric.corrwith(y, method='spearman')
# Filtramos las variables que tienen una correlación mayor a 0.2 con la variable target
spearman_corr_selected = spearman_corr[spearman_corr.abs() > 0.2]
spearman_corr_selected # variables importantes numéricas

Ya tenemos suficiente análisis sobre importancia de variables. En un futuro, cuando vayamos a hacer un modelo, deberíamos empezar haciendo dos modelos base (baseline), sin enriquecer. Estos modelos serían

seleccionando aquellos que tienen una relación extrictamente alta, como aquellos que ha devuelto spearman o Cramer; y
seleccionando todos los que no hemos eliminado previamenete en esta selección laxa.
Si el modelo 1 funciona igual o mejor que el modelo 2, este modelo sería el que deberíamos ajustar con hiperparámetros. Si el modelo 2 funciona mejor que el 1, entonces deberíamos vigilar de forma excepcional el ruido que pueden generar las variables que hemos eliminado. En este caso, deberíamos hacer un análisis de correlación entre las variables que hemos eliminado y las que hemos dejado. Si la correlación es alta, entonces deberíamos volver a incluirlas en el modelo y ajustar los hiperparámetros.

Último paso
Aquí volvemos a hacer un análisis bivariable con las columnas limpias que han quedado para sacar unas conclusiones con respecto al target. Dentro de estás variables hay algunas que muy probablemente sean eliminadas, como meal, children o baby. Por ahora no las eliminamos.

In [ ]:
df_clean.shape

In [ ]:
df_clean.head()

In [ ]:
df_clean.nunique()

In [ ]:
# Análisis bivariable de df_clean vs target
X = df_clean.drop(columns=['is_canceled']) # vector de características
y = df_clean['is_canceled'] # vector objetivo

# Hacemos visualizaciones personalizadas para las variables
fig, axs = plt.subplots(nrows=(len(X.columns) + 1) // 2, ncols=2, figsize=(15, 5 * ((len(X.columns) + 1) // 2)))

for i, column in enumerate(X.columns):
    row = i // 2
    col = i % 2
    if X[column].dtype == 'object':
        sns.countplot(data=df_clean, x=column, hue='is_canceled', ax=axs[row, col])
        axs[row, col].set_title(f'Distribución de {column} por is_canceled')
        axs[row, col].set_xlabel(column)
        axs[row, col].set_ylabel('Frecuencia')
        axs[row, col].tick_params(axis='x', rotation=45)
        axs[row, col].legend(title='is_canceled')
    elif X[column].dtype in ['int64', 'float64']:
        sns.kdeplot(data=df_clean[df_clean['is_canceled'] == 0], x=column, label='Not Canceled', fill=True, ax=axs[row, col], color='blue')
        sns.kdeplot(data=df_clean[df_clean['is_canceled'] == 1], x=column, label='Canceled', fill=True, ax=axs[row, col], color='red')
        axs[row, col].set_title(f'Densidad de {column} por is_canceled')
        axs[row, col].set_xlabel(column)
        axs[row, col].set_ylabel('Densidad')
        axs[row, col].legend()

# Eliminar ejes vacíos si hay
for j in range(i + 1, len(axs.flatten())):
    fig.delaxes(axs.flatten()[j])

plt.tight_layout()
plt.show()

Tras hacer el EDA, podemos ir sacando ciertas conclusiones de los datos:

Los city hotels tienen una tasa de cancelación más alta que los resort hotels.
A mayor número de días de antelación de la reserva, mayor tasa de cancelación.
El 2016 fue un año con mayor número de cancelaciones, pero también con un mayor número de reservas. Esta variable no aporará nada al modelo.
El mes de llegada al hotel no parece tener una relación directa con la tasa de cancelación.
Las semanas centrales del año tienen una mayor tasa de cancelación que las de los extremos. (Esto puede ser un indicativo de que los clientes cancelan más en verano y navidad)
El día del mes no parece tener una relación directa con la tasa de cancelación.
Las estancias más cortas de fines de semana tienen una menor tasa de cancelación, pero no es significativo.
Parece haber relación entre los días intra-semanales y la tasa de cancelación, pero no parece una relación visualmente clara.
Dos adultos tiene una taasa de cancelación más alta que una persona sola. Mientras que con los iños no parece que exista una relación clara.
Las parejas solteras cancelan más. Merece la pena investigar si existe correlación con la estacionalidad.
Se suele reservar solo con desayunos, mientras que pension completa es residual.
La variable de país necesita una agrupación, ya que tiene una alta cardinalidad. El país con más visitas es Portugal, con una tasa de cancelaciones muy alta. Reino Unido, España y Francia completan el top 3 de paises con más visitas, destacando RU con una tasa de cancelación baja. Parece ser un destino europeo del hemisferio norte, ya que concentra su actividad en el verano norte.
La variable de segmento del mercado es interesante, existiendo tipologías de clientes asociados a una mayor cancelación y viceversa. De forma similar parece que ocurre con el canal de distribución, aunque no parece tan claro.
Los clientes que repiten tienen una tasa de cancelación más baja. Lo que paerece indicar que el hotel tiene una buena experiencia de cliente.
Las habitaciones reservadas no parecen tener una relación clara con la tasa de cancelación.
Las reservas que no hacen cambios tienen una tasa de cancelación mayor.
Las reservas no reembolsables no sufren de cancelaciones, lo que indica que los clientes no están dispuestos a perder su dinero.
La variable de agente, pese a su alta cardinalidad, parece tener una relación clara con la tasa de cancelación.
Los clientes que han pasado por la lista de espera tienen mayor tasa de cancelación, lo que indica insatisfacción.